# Decision Trees

CART Book (1984): https://www.taylorfrancis.com/books/mono/10.1201/9781315139470/classification-regression-trees-leo-breiman





Soft Decision Trees (2012): https://arxiv.org/abs/1209.0422




NODE (2020): https://arxiv.org/abs/1909.06312

In [1]:
# Arabic Sentiment Analysis —  Decision Tree

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION

RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_clean.csv"
VAL_PATH      = "/content/val_clean.csv"
TEST_PATH     = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))
print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# DecisionTreeClassifier accepts sparse matrices directly
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])
print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

print("=" * 60)
print("  🌳 Training Optimized Decision Tree")
print("=" * 60)

clf = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=40,
    min_samples_split=10,      # prevent overfitting
    min_samples_leaf=5,        # smoother leaves
    max_features="sqrt",
    class_weight="balanced",   # handle imbalance
    random_state=RANDOM_STATE
)

clf.fit(X_train, y_train)

def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Decision Tree  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 10,264
Final feature shape (train) : (5067, 10266)

Classes : [0 1]  →  [0, 1]

  🌳 Training Optimized Decision Tree

────────────────────────────────────────────────────────────
  Decision Tree  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.6427
  Precision : 0.6708
  Recall    : 0.6427
  F1-Score  : 0.6274

  Classification Report:

              precision    recall  f1-score   support

           0       0.60      0.85      0.70       543
           1       0.74      0.44      0.55       543

    accuracy                           0.64      1086
   macro avg       0.67      0.64      0.63      1086
weighted avg       0.67      0.64      0.63      1086

  Confusion Matrix:
[[459  84]
 [304 239]]


────────────────────────────────────────────────────────

# Decision Tree (CART)

In [2]:
# Arabic Sentiment Analysis — CART Decision Tree (Breiman et al., 1984)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION

RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_clean.csv"
VAL_PATH      = "/content/val_clean.csv"
TEST_PATH     = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))
print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# DecisionTreeClassifier accepts sparse matrices directly
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])
print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — CART Decision Tree
# criterion='gini' : Gini impurity (CART default, slightly faster than entropy)
# max_depth        : caps tree depth to prevent overfitting on sparse TF-IDF
# min_samples_leaf : minimum samples per leaf — stronger regularisation
# class_weight     : handles class imbalance via weighted Gini
print("=" * 60)
print("  🌲 Training: CART Decision Tree (Breiman et al., 1984)")
print("=" * 60)

clf = DecisionTreeClassifier(
    criterion        = "gini",
    max_depth        = None,
    min_samples_leaf  = 4,
    max_features     = "sqrt",   # random feature subset per split (like RF)
    class_weight     = "balanced",
    random_state     = RANDOM_STATE,

)
clf.fit(X_train, y_train)
print(f"  Tree depth : {clf.get_depth()} | Leaves : {clf.get_n_leaves():,}\n")

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  CART Decision Tree  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 10,264
Final feature shape (train) : (5067, 10266)

Classes : [0 1]  →  [0, 1]

  🌲 Training: CART Decision Tree (Breiman et al., 1984)
  Tree depth : 186 | Leaves : 208


────────────────────────────────────────────────────────────
  CART Decision Tree  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7090
  Precision : 0.7090
  Recall    : 0.7090
  F1-Score  : 0.7090

  Classification Report:

              precision    recall  f1-score   support

           0       0.71      0.71      0.71       543
           1       0.71      0.71      0.71       543

    accuracy                           0.71      1086
   macro avg       0.71      0.71      0.71      1086
weighted avg       0.71      0.71      0.71      1086

  Confusion Matrix:
[[383 160]
 [156 387]]

# Soft Decision Tree

In [3]:
# Arabic Sentiment Analysis — Soft Decision Tree (Irsoy et al., 2012)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not found. Install with: pip install torch")
    print("   Falling back to sklearn DecisionTreeClassifier.\n")
    from sklearn.tree import DecisionTreeClassifier

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_clean.csv"
VAL_PATH      = "/content/val_clean.csv"
TEST_PATH     = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

# Soft Decision Tree hyperparameters
SDT_DEPTH      = 6        # depth D → 2^D leaves
SDT_LAMBDA     = 1e-3      # penalty on leaf distributions (entropy regulariser)
SDT_LR         = 5e-3       # Adam learning rate
SDT_EPOCHS     = 60
SDT_BATCH_SIZE = 128
LSA_DIM        = 256        # reduce TF-IDF to this dimension before SDT

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

np.random.seed(RANDOM_STATE)
if TORCH_AVAILABLE:
    torch.manual_seed(RANDOM_STATE)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji → LSA
print("=" * 60)
print("  ⚙️  Building Features (TF-IDF → LSA)")
print("=" * 60)

from sklearn.decomposition import TruncatedSVD

tfidf = TfidfVectorizer(**TFIDF_PARAMS)
X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

X_train_raw = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val_raw   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test_raw  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

# LSA dimensionality reduction (fit only on train)
lsa = TruncatedSVD(n_components=LSA_DIM, random_state=RANDOM_STATE)
X_train = lsa.fit_transform(X_train_raw).astype(np.float32)
X_val   = lsa.transform(X_val_raw).astype(np.float32)
X_test  = lsa.transform(X_test_raw).astype(np.float32)
print(f"Post-LSA feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
n_classes = len(le.classes_)
print(f"Classes : {le.classes_}  →  {list(range(n_classes))}\n")

# 4. MODEL — Soft Decision Tree (PyTorch)
if TORCH_AVAILABLE:

    class SoftDecisionTree(nn.Module):
        """
        Soft Decision Tree (Irsoy et al., 2012).
        Depth D tree with 2^D leaves. Each inner node is a sigmoid gate.
        Each leaf stores a class-probability distribution.
        """
        def __init__(self, in_features: int, n_classes: int, depth: int):
            super().__init__()
            self.depth     = depth
            self.n_leaves  = 2 ** depth
            self.n_inner   = self.n_leaves - 1   # inner nodes
            self.n_classes = n_classes

            # Inner node weights: each maps input → scalar gate probability
            self.inner_nodes = nn.Linear(in_features, self.n_inner, bias=True)
            # Leaf distributions: (n_leaves, n_classes) log-probabilities
            self.leaves = nn.Parameter(torch.zeros(self.n_leaves, n_classes))

        def forward(self, x: torch.Tensor):
            """Return class log-probabilities for each sample."""
            batch = x.size(0)

            # Gate probabilities for all inner nodes: (batch, n_inner)
            gate = torch.sigmoid(self.inner_nodes(x))

            # Traverse tree: compute probability of reaching each leaf
            # by multiplying gate decisions along the path
            # leaf_probs shape: (batch, n_leaves)
            leaf_probs = torch.ones(batch, 1, device=x.device)

            for d in range(self.depth):
                n_nodes_at_d = 2 ** d
                start_idx    = n_nodes_at_d - 1   # 0-indexed
                # Gates at this depth
                g = gate[:, start_idx : start_idx + n_nodes_at_d]  # (batch, n_nodes)
                # Each node splits into left (1-g) and right (g)
                left  = leaf_probs * (1 - g)
                right = leaf_probs * g
                leaf_probs = torch.cat([left, right], dim=1)  # (batch, 2*n_nodes)

            # leaf_probs: (batch, n_leaves)
            # leaf distributions: (n_leaves, n_classes) → softmax over classes
            leaf_dist = torch.softmax(self.leaves, dim=1)   # (n_leaves, n_classes)

            # Weighted sum over leaves: (batch, n_classes)
            out = torch.mm(leaf_probs, leaf_dist)
            return torch.log(out + 1e-9)

    print("=" * 60)
    print(f"  🌳 Training: Soft Decision Tree (depth={SDT_DEPTH})")
    print("=" * 60)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"   Device : {device}")

    model = SoftDecisionTree(
        in_features = LSA_DIM,
        n_classes   = n_classes,
        depth       = SDT_DEPTH,
    ).to(device)

    optimiser = optim.Adam(model.parameters(), lr=SDT_LR, weight_decay=1e-4)
    criterion = nn.NLLLoss()

    # DataLoaders
    def make_loader(X, y, shuffle=False):
        ds = TensorDataset(
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
        )
        return DataLoader(ds, batch_size=SDT_BATCH_SIZE, shuffle=shuffle)

    train_loader = make_loader(X_train, y_train, shuffle=True)

    for epoch in range(SDT_EPOCHS):
        model.train()
        total_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimiser.zero_grad()
            log_probs = model(Xb)
            # NLL loss + leaf entropy regularisation
            nll  = criterion(log_probs, yb)
            leaf_ent = -(
                torch.softmax(model.leaves, dim=1) *
                torch.log_softmax(model.leaves, dim=1)
            ).sum()
            loss = nll + SDT_LAMBDA * leaf_ent
            loss.backward()
            optimiser.step()
            total_loss += nll.item() * len(yb)
        if (epoch + 1) % 5 == 0:
            print(f"   Epoch {epoch+1:3d}/{SDT_EPOCHS}  |  NLL loss : {total_loss/len(y_train):.4f}")

    def sdt_predict(X_np):
        model.eval()
        with torch.no_grad():
            Xt = torch.tensor(X_np, dtype=torch.float32).to(device)
            log_probs = model(Xt)
        return log_probs.argmax(dim=1).cpu().numpy()

else:
    # Fallback to standard CART if PyTorch is unavailable
    clf = DecisionTreeClassifier(
        max_depth=20, min_samples_leaf=5, class_weight="balanced",
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)
    sdt_predict = clf.predict

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Soft Decision Tree  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  sdt_predict(X_val),  "Validation Set")
evaluate(y_test, sdt_predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features (TF-IDF → LSA)
Post-LSA feature shape (train) : (5067, 256)

Classes : [0 1]  →  [0, 1]

  🌳 Training: Soft Decision Tree (depth=6)
   Device : cpu
   Epoch   5/60  |  NLL loss : 0.6396
   Epoch  10/60  |  NLL loss : 0.5712
   Epoch  15/60  |  NLL loss : 0.5374
   Epoch  20/60  |  NLL loss : 0.5209
   Epoch  25/60  |  NLL loss : 0.5104
   Epoch  30/60  |  NLL loss : 0.5029
   Epoch  35/60  |  NLL loss : 0.4970
   Epoch  40/60  |  NLL loss : 0.4924
   Epoch  45/60  |  NLL loss : 0.4884
   Epoch  50/60  |  NLL loss : 0.4852
   Epoch  55/60  |  NLL loss : 0.4826
   Epoch  60/60  |  NLL loss : 0.4802

────────────────────────────────────────────────────────────
  Soft Decision Tree  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7689
  Precision : 0.7703
  Recall    : 0.7689
  F1